# Deep Learning:
 - CNN(Convolutional Neural Network) Algorithm
 - CNN for Image Classification

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

## Creating Datasets and DataLoaders:

In [2]:
# Datasets and DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale (0,1) => normalize (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

# CIFAR10() = a function/class that gives you images + their labels for training a neural network.

In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

## Converting our Datasets into DataLoaders:

In [4]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)3

## Build the CNN..

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32 ,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  # Kernel size = 2, stride =2

            # 2nd Layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            # 3rd Layer
            nn.Conv2d(64, 128, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)            
        )

        # Set Fully Connected Layer
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    # Start Forward Passing
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) #Flattening
        x = self.fc_layers(x)

        return x

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Train the CNN

In [13]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # Forward Propagration
        loss = criterion(output, labels) # Loss fnx calculation
        loss.backward() # Backward Propagation
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)} ")

epoch = 1/10 & loss=0.8499218198420752 
epoch = 2/10 & loss=0.6884085579830057 
epoch = 3/10 & loss=0.5672089943609884 
epoch = 4/10 & loss=0.46397774447413054 
epoch = 5/10 & loss=0.3750102592779852 
epoch = 6/10 & loss=0.2947749826399719 
epoch = 7/10 & loss=0.23223814676942117 
epoch = 8/10 & loss=0.1811837050873224 
epoch = 9/10 & loss=0.14400063531087412 
epoch = 10/10 & loss=0.12427239966057145 


## Evaluate our CNN

In [16]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {(correct_labels / total_labels) * 100:.2f}%")

Accuracy = 75.65%
